# CEG 论文结果包 Colab Pipeline

该 Notebook 用于在图像生成产物已经归档到 Google Drive 后, 在一个新的独立 Colab 会话中恢复图像生成阶段产物, 再调用仓库脚本完成 attack、真实 detection、fixed-FPR 校准、论文结果包导出和 Google Drive 归档。

重要边界:
- 本 Notebook 不读取上一个 Colab 会话的 `/content` 中间目录。
- 必须从 Google Drive 阶段 zip 恢复 `inputs/images`。
- 方法逻辑不写在 Notebook 中, 只调用仓库脚本。


该 Notebook 默认读取同一 `IMAGE_GENERATION_RUN_ID` 对应的 `external_baseline_outputs` 归档。如果已经运行 `colab_external_baseline_outputs.ipynb`, 不需要手动创建 baseline plan。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_CONTRACT_VERSION = "paper_results_pipeline_with_external_baseline_evidence_v3"
REPO_URL = "https://github.com/RICHAAARC/CEG.git"
REPO_BRANCH = ""
REPO_ROOT = Path("/content/CEG")
DRIVE_ROOT = Path("/content/drive/MyDrive/CEG")
LOCAL_RUNTIME_ROOT = Path("/content/ceg_runtime")

IMAGE_GENERATION_RUN_ID = "paper_main_probe_image_generation_outputs"
RUN_ID = f"{IMAGE_GENERATION_RUN_ID}_paper_results_package"
WORKSPACE = LOCAL_RUNTIME_ROOT / RUN_ID
RESET_LOCAL_RUNTIME_WORKSPACE = True

IMAGE_GENERATION_ARCHIVE = DRIVE_ROOT / "archives" / "image_generation_outputs" / f"{IMAGE_GENERATION_RUN_ID}.zip"
# 默认读取由 colab_external_baseline_outputs.ipynb 生成的同名 external baseline 归档。
# 如需临时跳过外部 baseline, 可以手动把 BASELINE_RUN_ID 改为空字符串。
BASELINE_RUN_ID = f"{IMAGE_GENERATION_RUN_ID}_external_baselines"
BASELINE_ARCHIVE = (DRIVE_ROOT / "archives" / "external_baseline_outputs" / f"{BASELINE_RUN_ID}.zip") if BASELINE_RUN_ID else None

RUN_PIPELINE = True
ALLOW_INCOMPLETE_PACKAGE = True
ALLOW_INVALID_ARCHIVE = True
ATTACK_FAMILIES = "brightness_contrast,gaussian_noise,rotate,resize,jpeg"
PERSPECTIVE_OFFSETS = "0.0,0.04,-0.04"
LOCAL_DEFORMATION_ENABLED = "true"
TARGET_FPR = 0.01
ATTESTATION_KEY_ENV = "CEG_ATTESTATION_KEY"
ATTESTATION_KEY_ID = "formal_colab_run_key"
SEMANTIC_MASK_BACKEND = "ceg_inspyrenet_semantic_mask"
PREPARE_INSPYRENET_WEIGHT = True
INSPYRENET_WEIGHT_URL = "https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth"
INSPYRENET_WEIGHT_DRIVE_PATH = Path("/content/drive/MyDrive/Models/inspyrenet/ckpt_base.pth")
BASELINE_OBSERVATIONS = WORKSPACE / "external_baselines" / "baseline_observations.json"
BASELINE_EXECUTION_MANIFEST = WORKSPACE / "external_baselines" / "baseline_execution_manifest.json"
# 默认不把外部 baseline 导入声明为正式结果。正式声明必须由前序 baseline 自身证据链决定。
BASELINE_FORMAL_RESULT_CLAIM = False


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"非 Colab 环境或 Drive 已挂载: {exc}")

if not REPO_ROOT.exists():
    cmd = ["git", "clone"]
    if REPO_BRANCH:
        cmd += ["--branch", REPO_BRANCH]
    cmd += [REPO_URL, str(REPO_ROOT)]
    print("运行:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--all", "--prune"], check=True)
    if REPO_BRANCH:
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transparent-background"], check=True)

from paper_workflow.colab_utils.runtime import (
    ensure_attestation_key,
    extract_stage_archive,
    prepare_inspyrenet_weight,
    rewrite_image_pairs_for_restored_archive,
)

if WORKSPACE.exists() and RESET_LOCAL_RUNTIME_WORKSPACE:
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True, exist_ok=True)

if not IMAGE_GENERATION_ARCHIVE.is_file():
    raise FileNotFoundError(f"图像生成阶段归档不存在, 请先运行 colab_pilot_image_generation_outputs.ipynb: {IMAGE_GENERATION_ARCHIVE}")
extract_stage_archive(
    archive_zip_path=IMAGE_GENERATION_ARCHIVE,
    destination_root=WORKSPACE / "inputs" / "images",
    reset=True,
)

path_rewrite_report = rewrite_image_pairs_for_restored_archive(
    image_pairs_path=WORKSPACE / "inputs" / "images" / "image_pairs.json",
    image_output_root=WORKSPACE / "inputs" / "images",
)
print(json.dumps(path_rewrite_report, ensure_ascii=False, indent=2))
if path_rewrite_report["overall_decision"] != "pass":
    raise RuntimeError("恢复后的 image_pairs 路径重写失败, 请检查图像归档内容。")

if BASELINE_ARCHIVE is not None and BASELINE_ARCHIVE.is_file():
    extract_stage_archive(
        archive_zip_path=BASELINE_ARCHIVE,
        destination_root=WORKSPACE / "external_baselines",
        reset=True,
    )
else:
    print("未配置或未找到 baseline 阶段归档, 本次结果包将不读取外部 baseline 归档。")

ensure_attestation_key(ATTESTATION_KEY_ENV)
if PREPARE_INSPYRENET_WEIGHT:
    prepare_inspyrenet_weight(
        drive_weight_path=INSPYRENET_WEIGHT_DRIVE_PATH,
        weight_url=INSPYRENET_WEIGHT_URL,
    )

print("notebook_contract_version =", NOTEBOOK_CONTRACT_VERSION)
print("image_generation_archive =", IMAGE_GENERATION_ARCHIVE)
print("baseline_run_id =", BASELINE_RUN_ID)
print("baseline_archive =", BASELINE_ARCHIVE, "exists=", BASELINE_ARCHIVE.is_file() if BASELINE_ARCHIVE is not None else False)
print("workspace =", WORKSPACE)
print("image_pairs =", WORKSPACE / "inputs" / "images" / "image_pairs.json")
print("baseline_observations =", BASELINE_OBSERVATIONS, "exists=", BASELINE_OBSERVATIONS.exists())
print("baseline_execution_manifest =", BASELINE_EXECUTION_MANIFEST, "exists=", BASELINE_EXECUTION_MANIFEST.exists())
print("baseline_formal_result_claim =", BASELINE_FORMAL_RESULT_CLAIM)


In [ ]:
cmd = [
    sys.executable,
    "scripts/run_colab_paper_results_pipeline.py",
    "--workspace", str(WORKSPACE),
    "--drive-root", str(DRIVE_ROOT),
    "--run-id", RUN_ID,
    "--image-pairs", str(WORKSPACE / "inputs" / "images" / "image_pairs.json"),
    "--attack-families", ATTACK_FAMILIES,
    "--target-fpr", str(TARGET_FPR),
    "--attestation-key-env", ATTESTATION_KEY_ENV,
    "--attestation-key-id", ATTESTATION_KEY_ID,
    "--semantic-mask-backend", SEMANTIC_MASK_BACKEND,
    "--detection-formal-result-claim",
    "--perspective-offsets", PERSPECTIVE_OFFSETS,
    "--local-deformation-enabled", LOCAL_DEFORMATION_ENABLED,
]
if BASELINE_OBSERVATIONS.exists():
    cmd += ["--baseline-observations", str(BASELINE_OBSERVATIONS)]
    if BASELINE_EXECUTION_MANIFEST.exists():
        cmd += ["--baseline-evidence-path", str(BASELINE_EXECUTION_MANIFEST)]
    if BASELINE_FORMAL_RESULT_CLAIM:
        cmd.append("--baseline-formal-result-claim")
if ALLOW_INCOMPLETE_PACKAGE:
    cmd.append("--allow-incomplete-package")
if ALLOW_INVALID_ARCHIVE:
    cmd.append("--allow-invalid-archive")
print("运行:", " ".join(cmd))

def print_paper_pipeline_failure_reports():
    report_paths = [
        WORKSPACE / "paper_results_pipeline" / "colab_paper_results_pipeline_manifest.json",
        WORKSPACE / "paper_results_pipeline" / "attack_outputs" / "image_manifests" / "attack_shard_manifest.json",
        WORKSPACE / "paper_results_pipeline" / "detection_outputs" / "ceg_detection_producer_manifest.json",
        WORKSPACE / "paper_results_pipeline" / "metric_outputs" / "metric_execution_manifest.json",
        WORKSPACE / "paper_results_pipeline" / "calibrated_paper_results_package" / "paper_results_package" / "paper_results_package_manifest.json",
    ]
    for path in report_paths:
        print("报告:", path, "exists=", path.exists())
        if not path.is_file():
            continue
        text = path.read_text(encoding="utf-8", errors="replace")
        print(text[-8000:])

if RUN_PIPELINE:
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError:
        print_paper_pipeline_failure_reports()
        raise
else:
    print("RUN_PIPELINE = False, 仅打印命令。")


In [ ]:
paths = {
    "image_generation_archive": IMAGE_GENERATION_ARCHIVE,
    "local_runtime_workspace": WORKSPACE,
    "image_pairs": WORKSPACE / "inputs" / "images" / "image_pairs.json",
    "pipeline_manifest": WORKSPACE / "paper_results_pipeline" / "colab_paper_results_pipeline_manifest.json",
    "paper_results_package": WORKSPACE / "paper_results_pipeline" / "calibrated_paper_results_package" / "paper_results_package",
    "drive_archives": DRIVE_ROOT / "package_archives",
}
for name, path in paths.items():
    print(name, "=", path, "exists=", path.exists())
